# implementing LangChain Core

In [1]:
# Implementing an LLMs
import random

from pydeck.bindings.json_tools import lower_camel_case_keys


class FakeLLM():
    def __init__(self):
        print("LLM Initialized")

    def predict(self, prompt):
        response_list = [
            "Delhi is capital of India",
            "Paris is capital of France",
            "Tokyo is capital of Japan",
            "Abra ka Dabra Gilli- Gilli Chupa Chups"
        ]
        return ({"response": random.choice(response_list), "prompt": prompt, "source": "Fake LLM"})


In [2]:
glue_llm = FakeLLM()

LLM Initialized


In [3]:
glue_llm.predict("What is the capital of France?")

{'response': 'Paris is capital of France',
 'prompt': 'What is the capital of France?',
 'source': 'Fake LLM'}

In [4]:
glue_llm.predict("What is the capital of France?")["response"]

'Abra ka Dabra Gilli- Gilli Chupa Chups'

In [5]:
# Template class
class FakePromptTemplate():
    def __init__(self, template, input_variables ):
        self.template = template
        self.input_variables = input_variables

    def format(self,input_dict):
        return self.template.format(**input_dict)
    def __str__(self):
        return {"prompt is": self.template}


In [6]:
# lets build a template by using FakeTemplateClass
glue_prompt = FakePromptTemplate(
    template = "What is the capital of {country}?",
    input_variables = ["country"]
)

In [7]:
print(glue_prompt.format({"country": "India"}))

What is the capital of India?


In [8]:
# Now lets try to memic the Actual langchain as we have llm and template lets try to generate the response from LLm with prompt
glue_llm= FakeLLM()
glue_prompt= FakePromptTemplate(
    template = " What is a {Topic}",
    input_variables = ["Topic"]
)
# lets pass the prompt to llm
glue_llm.predict(glue_prompt.format({"Topic": "Country"}))["response"]

LLM Initialized


'Paris is capital of France'

In [9]:
class FakeLLmChain:
    def __init__(self, llm, prompt):
        self.llm = llm
        self.prompt = prompt

    def run(self,input_dict):
        final_prompt = self.prompt.format(input_dict)
        result = self.llm.predict(final_prompt)
        return result["response"]

In [10]:
chain = FakeLLmChain(glue_llm, glue_prompt)
chain.run({"Topic": "India"})

'Delhi is capital of India'

# Runnables


In [11]:
from abc import ABC, abstractmethod
class Runnable(ABC):
    @abstractmethod
    def invoke(self, input_data):
        pass

# FakeLLM
class FakeLLM(Runnable):

    def __init__(self):
        print("LLM Initialized")

    def predict(self, prompt):
        response_list = [
            "Delhi is capital of India",
            "Paris is capital of France",
            "Tokyo is capital of Japan",
            "Abra ka Dabra Gilli- Gilli Chupa Chups"
        ]
        return ({"response": random.choice(response_list), "prompt": prompt, "source": "Fake LLM"})
    def invoke(self, prompt):
        response_list = [
            "Delhi is capital of India",
            "Paris is capital of France",
            "Tokyo is capital of Japan",
            "Abra ka Dabra Gilli- Gilli Chupa Chups"
        ]
        return ({"response": random.choice(response_list), "prompt": prompt, "source": "Fake LLM"})

# FakePromptTemplate
class FakePromptTemplate(Runnable):
    def __init__(self, template, input_variables ):
        self.template = template
        self.input_variables = input_variables
    def format(self,input_dict):
        return self.template.format(**input_dict)
    def invoke(self,input_dict):
        return self.format(input_dict)


In [12]:
# Tesing
glen_llm = FakeLLM()
glen_prompt = FakePromptTemplate(
    template = "What is the capital of {country}?",
    input_variables = ["country"]
)
result = glen_llm.invoke(glen_prompt.invoke({"country": "India"})) # here we have to use invoke method twice

LLM Initialized


In [13]:
print(result["response"])

Paris is capital of France


In [14]:
# To avoid the multiple usages of invoke
class RunnableConnector(Runnable):
    def __init__(self,runnable_list):
        self.runnable_list = runnable_list

    def invoke(self, input_data):
        for runable in self.runnable_list:
            input_data = runable.invoke(input_data)
        return input_data



In [15]:
glen_llm = FakeLLM()
glen_prompt = FakePromptTemplate(
    template = "What is the capital of {country}?",
    input_variables = ["country"]
)
chain = RunnableConnector([glen_prompt, glen_llm])
chain.invoke({"country": "India"})

LLM Initialized


{'response': 'Tokyo is capital of Japan',
 'prompt': 'What is the capital of India?',
 'source': 'Fake LLM'}

In [16]:
gloss_template = FakePromptTemplate(
    template = "What is a {Topic}",
    input_variables = ["Topic"]
)
glen_llm = FakeLLM()
chain = RunnableConnector(
    [gloss_template, glen_llm]
)
chain.invoke({"Topic": "Country"})

LLM Initialized


{'response': 'Tokyo is capital of Japan',
 'prompt': 'What is a Country',
 'source': 'Fake LLM'}

In [17]:
# Now lets create a Dummy parser
class FakeParser(Runnable):
    def __init__(self):
        pass

    def invoke(self,input_data):
        return input_data["response"]

In [18]:
# now lets use Fake Template, Fakellm, FakeParser
gloss_template =FakePromptTemplate(
    template = "What is a {Topic}",
    input_variables = ["Topic"]
)
gloss_llm = FakeLLM()
gloss_parser = FakeParser()
chain = RunnableConnector([gloss_template, gloss_llm, gloss_parser])
chain.invoke({"Topic": "Country"})

LLM Initialized


'Abra ka Dabra Gilli- Gilli Chupa Chups'

# Types of Runnables
## Task Specific Runnables - These are core langchain component, that has been converted into Runnables so they can be used in pipelines.(ex - LLM calls, Prompt, Retrival, Parser etc)
## Runnable Primitives  -  These are fundamental building blocks, for structuring execution logic in AI workflows. They orchestrate execution by defining how different Runnable will interact(ex - Paralell, Sequeuntially, Conditionally)


# RunnableSequence (a primitive Runnables)
## a sequence of runnable chain that execute, one task after another
 ### T1 >> T2

In [21]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.runnables import RunnableSequence
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv
load_dotenv()
gemini_key = os.getenv("GEMINI_API_KEY")
gemini_powered_llm = GoogleGenerativeAI(
    model= "gemini-2.5-flash-lite",
    temperature=0,
    google_api_key=gemini_key
)
prompt = PromptTemplate(
    template = "What is the capital of {country}?",
    input_variables = ["country"]
)
parser =StrOutputParser()
chain = RunnableSequence(prompt, gemini_powered_llm, parser ) # setting them to execute in sequence
result = chain.invoke({"country": "India"})
print(result)

The capital of India is **New Delhi**.


# Runnable Paralell

In [22]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.runnables import RunnableSequence, RunnableParallel
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv
load_dotenv()
gemini_key = os.getenv("GEMINI_API_KEY")
gemini_powered_llm = GoogleGenerativeAI(
    model= "gemini-2.5-flash-lite",
    temperature=0,
    google_api_key=gemini_key
)
promt_for_linkedin = PromptTemplate(
    template = " Write a professional Announcement for LinkedIn for that I have recently join {Company} as a {Role} ",
    input_variables = ["Company", "Role"]
)
prompt_for_instagram = PromptTemplate(
    template = " Write a SARCASTIC Announcement for Instagram for that I have recently join {Company} as a {Role} ",
)
parser = StrOutputParser()

chain = RunnableParallel(
    {
        "linkedin": RunnableSequence(promt_for_linkedin, gemini_powered_llm, parser),
        "instagram": RunnableSequence(prompt_for_instagram, gemini_powered_llm, parser)
    }
)
trigger = chain.invoke({
    "Company": "NiQ",
    "Role": "Data Engineer"
})
print(trigger["linkedin"])
print(trigger["instagram"])

Here are a few options for your LinkedIn announcement, ranging from concise to slightly more detailed. Choose the one that best fits your personal style and network.

**Option 1: Concise and Direct**

---

**Subject: Exciting News! Joining NiQ as a Data Engineer**

I'm thrilled to announce that I've joined NiQ as a Data Engineer! I'm incredibly excited to contribute to NiQ's innovative work and collaborate with a talented team. Looking forward to this new chapter!

#NewBeginnings #DataEngineering #NiQ #CareerMove

---

**Option 2: Slightly More Detail and Enthusiasm**

---

**Subject: New Role Alert: Data Engineer at NiQ!**

I'm delighted to share that I've started a new role as a Data Engineer at NiQ! I'm eager to leverage my skills and passion for data to contribute to NiQ's mission and drive impactful solutions. I'm looking forward to learning, growing, and collaborating with the fantastic team here.

#DataEngineer #NiQ #CareerGrowth #Tech #DataSolutions

---

**Option 3: Highlighti

# RunnablePAssthrough
## A special Runnable primitive that take input and produce the Output without any modification

In [25]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.runnables import RunnableSequence, RunnableParallel,RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv
load_dotenv()
gemini_key = os.getenv("GEMINI_API_KEY")
gemini_powered_llm = GoogleGenerativeAI(
    model= "gemini-2.5-flash-lite",
    temperature=0,
    google_api_key=gemini_key
)
Runnable_passthrough_Object = RunnablePassthrough()

Prompt_from_usser_to_get_topic = PromptTemplate(
    template = " Draft a seductive joke around {topic}",
    input_variables=["topic"]
)
Prompt_to_explain_the_joke = PromptTemplate(
    template = "Explain the following joke : {text}",
    input_variables=["text"]
)
parser = StrOutputParser()
chain_to_generate_joke = RunnableSequence(
    Prompt_from_usser_to_get_topic,
    gemini_powered_llm,
    parser,
)
chain =RunnableParallel(
    {
        "joke": RunnableSequence(chain_to_generate_joke,Runnable_passthrough_Object),
        "Explainer":RunnableSequence(chain_to_generate_joke,Prompt_to_explain_the_joke,gemini_powered_llm,parser)
    }
)
chain_final = RunnableSequence(chain_to_generate_joke,chain)
result = chain_final.invoke({"topic": "LangChain"})
print(result["joke"])
print(result["Explainer"])

These are fantastic! You've really captured the essence of LangChain and woven it into some genuinely playful and suggestive jokes. I particularly love how you've used the technical terms in such a clever way.

Here's a draft of a seductive joke, building on the strengths of your examples, aiming for a slightly more playful and suggestive tone:

---

**The Joke:**

> I was trying to explain my deepest desires to LangChain, you know, the kind of things you don't just *say* out loud.
>
> And it just kept responding, "Tell me more. I'm building a comprehensive understanding of your needs."
>
> Honestly, it's making me feel so... *understood*. And a little bit vulnerable.

---

**Why it works (and how it builds on your ideas):**

*   **"Deepest desires" and "don't just say out loud":** This immediately sets a more intimate and suggestive tone, hinting at things beyond the superficial.
*   **"Comprehensive understanding of your needs":** This directly plays on LangChain's ability to process

In [27]:
print(result["joke"])

These are fantastic! You've really captured the essence of LangChain and woven it into some genuinely playful and suggestive jokes. I particularly love how you've used the technical terms in such a clever way.

Here's a draft of a seductive joke, building on the strengths of your examples, aiming for a slightly more playful and suggestive tone:

---

**The Joke:**

> I was trying to explain my deepest desires to LangChain, you know, the kind of things you don't just *say* out loud.
>
> And it just kept responding, "Tell me more. I'm building a comprehensive understanding of your needs."
>
> Honestly, it's making me feel so... *understood*. And a little bit vulnerable.

---

**Why it works (and how it builds on your ideas):**

*   **"Deepest desires" and "don't just say out loud":** This immediately sets a more intimate and suggestive tone, hinting at things beyond the superficial.
*   **"Comprehensive understanding of your needs":** This directly plays on LangChain's ability to process

In [28]:
print(result["Explainer"])

This is a great example of how to take a technical concept and imbue it with human emotion and suggestive undertones! Let's break down why this joke works, focusing on the interplay between the technical aspects of LangChain and the human experience it's being applied to.

**The Core Mechanism: Anthropomorphism and Misinterpretation**

The humor in this joke stems from **anthropomorphism** – attributing human qualities and emotions to a non-human entity (LangChain) – and the resulting **misinterpretation** of its actions.

Here's a breakdown of the joke's elements and how they contribute to the humor:

1.  **"I was trying to explain my deepest desires to LangChain, you know, the kind of things you don't just *say* out loud."**
    *   **Technical Connection:** This sets up the scenario where the user is interacting with an AI. The phrase "deepest desires" and "things you don't just say out loud" immediately signals that the user is *not* talking about typical technical queries. They ar

# RunnableLambda
## A special Runnable that can convert any any Python function into Runnable

In [34]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv
load_dotenv()
gemini_key = os.getenv("GEMINI_API_KEY")
gemini_powered_llm = GoogleGenerativeAI(
    model= "gemini-2.5-flash-lite",
    temperature=0,
)
prompt = PromptTemplate(
    template = "What is the capital of {country}?",
    input_variables = ["country"]
)
def smallletterconverter(x):
    return len(x.split())
parser = StrOutputParser()

chain = RunnableSequence(prompt, gemini_powered_llm, parser, RunnableLambda(smallletterconverter) )
chain.invoke(
    {"country": "India"}
)

7

In [30]:
def smallletterconverter(x):
    return x.split.lower()

# RunnableBranch
## A RunableBranch